In [ ]:
from typing import (Union,
                    Callable,
                    List,
                    Dict,
                    Tuple)
import json
import numpy as np
import pandas as pd
from copy import deepcopy
from transformers import (set_seed,
                          PreTrainedTokenizer,
                          BertTokenizer,
                          BertModel, 
                          DistilBertTokenizer,
                          DistilBertModel)
# Configs
from transformers import (BertConfig, 
                          DistilBertConfig)

from torch.utils.data import (Dataset,
                              DataLoader)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import (Optimizer,
                         Adadelta,
                         AdamW)
from torch.optim.lr_scheduler import StepLR

from sklearn.dummy import DummyClassifier

from tqdm.auto import tqdm
from time import perf_counter
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_context("notebook")
sns.set_style("white")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
with open("../input/clinc150-dataset/data/data_full.json") as json_file:
    data = json.load(json_file)
print(data.keys())

In [ ]:
def parse_data(data: Dict) -> pd.DataFrame:
    """
    Reconstructs a Pandas DataFrame from the CLINC150 dataset.
    """
    df = pd.DataFrame()
    for k in data.keys():
        split_df = pd.DataFrame(data[k], columns=["text", "intent"])
        split_df.insert(0, "split", k)
        df = pd.concat([df, split_df], axis=0, ignore_index=True)
    return df

df = parse_data(data)
df.head()

In [ ]:
# Split into 'train', 'test'
train_df = df.loc[df["split"].apply(lambda x: x.split("_")[-1]) == "train"].copy(deep=True)
eval_df = df.loc[df["split"].apply(lambda x: x.split("_")[-1]) == "val"].copy(deep=True)
test_df = df.loc[df["split"].apply(lambda x: x.split("_")[-1]) == "test"].copy(deep=True)

print(train_df.shape)
print(eval_df.shape)
print(test_df.shape)

In [ ]:
# All classes
classes = train_df["intent"].unique().tolist()
classes[:20]

In [ ]:
# Sorting labels alphabetically and putting 'oos' at last position
classes = [l for l in classes if l != "oos"]

# Remove duplicates (if there're any)
classes_sort = set(sorted(classes) + ["oos"])
classes_sort = list(classes_sort)
classes_sort[-10:]

In [ ]:
# Number of classes
n_classes = len(classes_sort)
print(f"Amount of labels: {n_classes}")

In [ ]:
# Encode classes
class2id = {c: idx for idx, c in enumerate(classes_sort)}
# Print first 10 
{k: v for k, v in class2id.items() if v in list(range(10))}

In [ ]:
# Encode data
train_df["labels"] = train_df["intent"].map(class2id)
eval_df["labels"] = eval_df["intent"].map(class2id)
test_df["labels"] = test_df["intent"].map(class2id)

# Convert all labels to int
train_df["labels"] = train_df["labels"].astype(int)
eval_df["labels"] = eval_df["labels"].astype(int)
test_df["labels"] = test_df["labels"].astype(int)

# Drop unused columns
train_df.drop(columns=["split", "intent"], inplace=True)
eval_df.drop(columns=["split", "intent"], inplace=True)
test_df.drop(columns=["split", "intent"], inplace=True)

train_df.head()

In [ ]:
clf = DummyClassifier()

# Get train/evaluation data
X_train, y_train = train_df["text"].tolist(), train_df["labels"].tolist()
X_eval, y_eval = eval_df["text"].tolist(), eval_df["labels"].tolist()

# Train the classifier
clf.fit(X=X_train, y=y_train)

# Generate predictions and calculate the accuracy
y_preds = clf.predict(X_eval)
baseline_acc = accuracy_score(y_eval, y_preds)
print(f"Baseline accuracy: {baseline_acc:.3f}")

In [ ]:
def train_one_epoch(model: nn.Module, 
                    dataloader: DataLoader, 
                    criterion: Union[nn.Module, Callable], 
                    optimizer: Optimizer,
                    device: Union[None, str, int]=None, 
                    seed: Union[None, int, str]=None) -> float:
    """
    Train model for one whole epoch and return
    the mean loss per train batch.
    """
    if seed is not None:
        if isinstance(seed, int):
            set_seed(seed)
        else:
            raise TypeError("seed must be an integer.")
            
    if device is not None:
        device = device
    else: # Infer
        device = "cuda" if torch.cuda.is_available() else "cpu"
    
    model.train()
    model = model.to(device)
    train_loss = 0
    for batch in tqdm(dataloader, desc="train", leave=False):
        # Zero gradients
        optimizer.zero_grad()
        
        # Send data to device
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        labels = batch["labels"].to(device)
        
        # Run forward pass
        outputs = model(**inputs)
        
        # Calculate loss
        loss = criterion(outputs, labels)
        train_loss += loss.item()
        
        # Calculate gradients
        loss.backward()
        
        # Update weigths
        optimizer.step()
    
    # Return mean loss per batch
    return train_loss/len(dataloader)

In [ ]:
def evaluate(model: nn.Module, 
             dataloader: DataLoader,
             criterion: Union[nn.Module, Callable],
             device: Union[None, int, str]=None) -> Tuple:
    """
    Run an evaluation pass and return the mean loss
    per evaluation batch and total accuracy.
    """
    if device is not None:
        device = device
    else: # Infer
        device = "cuda" if torch.cuda.is_available() else "cpu"
        
    # Freeze dropout layer
    model.eval()
    model = model.to(device)
    eval_loss = 0
    true_labels, all_preds = [], []
    
    for batch in tqdm(dataloader, desc="eval", leave=False):
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        labels = batch["labels"].to(device)
        
        # Run forward pass without tracking grads
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Calculate loss
        eval_loss += criterion(outputs, labels).item()
        
        # Apply Softmax to normalize multi-class logits to predictions
        probs = F.softmax(outputs, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        
        # Store data to calculate accuracy
        all_preds.extend(preds.tolist())
        true_labels.extend(labels.tolist())
    
    # Calculate accuracy
    eval_acc = accuracy_score(true_labels, all_preds)
    # Return mean loss per batch
    return (eval_loss/len(dataloader), eval_acc)

In [ ]:
def train(model: nn.Module, 
          train_loader: DataLoader, 
          eval_loader: DataLoader,
          criterion: Union[nn.Module, Callable],
          optimizer: Optimizer,
          n_epochs: int=1,
          scheduler: Union[None, Optimizer]=None,
          device: Union[None, str, int]=None,
          seed: Union[None, int, str]=None) -> Dict[str, List[float]]:
    """
    Train the model for n_epochs and evaluate on the
    evaluation set and returns the training history dict.
    
    The training saves the model's state_dict after every
    epoch and at the end loads the best one.
    """
    if seed is not None:
        if isinstance(seed, int):
            set_seed(seed)
        else:
            raise TypeError("seed must be an integer.")
            
    if device is not None:
        device = device
    else: # Infer
        device = "cuda" if torch.cuda.is_available() else "cpu"
        
    # History
    history = {"train_loss": [], "eval_loss": [], "eval_acc": []}
    
    # Save best model
    best_acc = 0
    best_checkpoint = dict()
        
    for epoch in tqdm(range(1, n_epochs + 1), desc="train"): 
        # Train one epoch and calculate the train time in seconds
        start_time = perf_counter()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, seed)
        end_time = perf_counter()
        elapsed_time = end_time - start_time
        
        # Adjust lr
        if scheduler is not None:
            scheduler.step()

        # Run evaluation
        eval_loss, eval_acc = evaluate(model, eval_loader, criterion, device)
        
        # Save best model
        is_best = eval_acc > best_acc
        if is_best:
            # Save model's state_dict
            model.eval() # Freeze dropout
            best_checkpoint["state_dict"] = deepcopy(model.state_dict())
            # Update best eval accuracy
            best_acc = eval_acc
            
        # Updated history
        history["train_loss"].append(train_loss)
        history["eval_loss"].append(eval_loss)
        history["eval_acc"].append(eval_acc)
        
        # Report
        print(f"Epoch {epoch}: Train Loss {train_loss:.3f} - Eval Loss {eval_loss:.3f} - Eval Acc {eval_acc:.3f} - Train Time {elapsed_time:,.1f}")
        
    print("Loading best checkpoint...")
    model = model.load_state_dict(best_checkpoint["state_dict"])
    
    return history

In [ ]:
def plot_train_history(history: Dict[str, List[float]]) -> None:
    """
    Plot train and evaluation loss and evaluation accuracy.
    """
    fig, axes = plt.subplots(ncols=2, figsize=(10, 4))
    train_loss = np.array(history["train_loss"])
    eval_loss = np.array(history["eval_loss"])
    eval_acc = np.array(history["eval_acc"])
    epochs = np.arange(1, len(train_loss) + 1, 1, dtype=int)
    
    # Plot loss
    sns.lineplot(x=epochs, y=train_loss, label="train loss", ax=axes[0])
    sns.lineplot(x=epochs, y=eval_loss, label="eval loss", ax=axes[0])
    axes[0].set_title("Train and Evaluation Loss")
    axes[0].set_ylabel("Loss")
    axes[0].set_xlabel("epochs")
    
    # Plot accuracy
    sns.lineplot(x=epochs, y=eval_acc, label="eval accuracy", ax=axes[1])
    axes[1].set_title("Evaluation Accuracy")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_xlabel("epochs")
    fig.tight_layout()
    plt.show()

In [ ]:
def get_num_parameters(model: nn.Module, count_nonzero_only: bool=False) -> int:
    """
    Calculate the number of parameters of the model.
    If count_nonzero_only is True, count only nonzero weights.
    """
    num_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_elements += param.count_nonzero()
        else:
            num_elements += param.numel()
    return num_elements

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(checkpoint)
bert_model = BertModel.from_pretrained(checkpoint).to(device)

In [ ]:
n_params = get_num_parameters(bert_model)
print(f"BERT Model has {n_params:,} parameters.")

In [ ]:
class BERTForClassification(nn.Module):
    """
    BERT model with classification head.
    
    This class extends a BERT model with a custom classification head,
    allowing it to be fine-tuned for classification tasks.
    The `pretrainedModel` is a pre-trained BERT model, and `n_labels`
    specifies the number of distinct labels for classification.

    Parameters:
        pretrainedModel (BertModel): A pre-trained BERT model.
        n_labels (int): The number of distinct labels for classification.
    """
    def __init__(self, 
                 pretrainedModel: BertModel,
                 n_labels: int):
        super(BERTForClassification, self).__init__()
        self.config = pretrainedModel.config
        self.hidden_size = self.config.hidden_size
        self.n_labels = n_labels
        
        # Pretrained BERT
        self.bert = pretrainedModel
        
        # Get config dropout rate
        if self.config.classifier_dropout is not None:
            classifier_dropout = self.config.classifier_dropout
        elif self.config.hidden_dropout_prob is not None:
            classifier_dropout = self.config.hidden_dropout_prob
        else:
            classifier_dropout = 0.1

        # Linear head (dropout + linear layer)
        self.dropout = nn.Dropout(classifier_dropout)
        self.linear = nn.Linear(self.hidden_size, self.n_labels)
        
    
    def forward(self, **kwargs):
        """
        Perform a forward pass for the BERT model with the classification head.

        Parameters:
            **kwargs: Keyword arguments to pass to the BERT model.
                May include the following keys 'input_ids', 'attention_mask' and
                'token_type_ids' in case of pretrained BERT.

        Returns:
            torch.Tensor: The output tensor after passing through the classification head.
        """
        outputs = self.bert(**kwargs).get("pooler_output")
        outputs = self.dropout(outputs)
        outputs = self.linear(outputs)
        return outputs

In [ ]:
class ClincDataset(Dataset):
    """
    A custom PyTorch dataset for the Clinc Intent Classification task.

    Args:
        text (List[str]): List of text samples.
        labels (List[int]): List of corresponding labels.
    """
    def __init__(self, 
                 text: List[str],
                 labels: List[int]):
        self.text = text
        self.labels = labels
        
    
    def __len__(self):
        return len(self.text)
    
    
    def __getitem__(self, idx):
        return self.text[idx], self.labels[idx]
    
train_set = ClincDataset(*zip(*train_df.values.tolist()))
eval_set = ClincDataset(*zip(*eval_df.values.tolist()))
test_set = ClincDataset(*zip(*test_df.values.tolist()))

train_set[:2]

In [ ]:
def create_collate_fn(tokenizer: PreTrainedTokenizer) -> Callable:
    """
    Returns a collate function that utilizes the provided tokenizer.
    
    Parameters:
        tokenizer (PreTrainedTokenizer): The tokenizer to be used.
    
    Returns:
        Callable: The custom collate function.
    """
    def collate_fn(inputs: List[Tuple]) -> Dict[str, torch.tensor]:
        """
        Custom collate function to tokenize and pad inputs and return labels.
        
        Parameters:
            inputs (List[Tuple]): A list of input text and labels.
            
        Returns:
            Dict[str, torch.Tensor]: A dictionary containing tokenized and 
                padded inputs with labels.
        """
        # Unpack text and labels
        text, labels = zip(*inputs)
        # Create input_ids, token_type_ids and attention_mask
        inputs = tokenizer(text, 
                           truncation=True,
                           padding=True,
                           max_length=128,
                           return_tensors="pt")

        # Create labels tensor
        labels = torch.tensor(labels)
        # Insert labels
        inputs.update({"labels": labels})
        return inputs
    return collate_fn

collate_fn = create_collate_fn(tokenizer)

In [ ]:
# Set seed for reproductivity
set_seed(123)

# Create dataloaders
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval_set, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [ ]:
# Create model
model = BERTForClassification(bert_model, n_classes)

# Training args
n_epochs = 2 # 8
lr = 3e-5 # learning rate
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=lr)
scheduler = StepLR(optimizer, step_size=1, gamma=.9) # Reduces 10% of the initial lr

# Train the model
history = train(model,
                train_loader,
                eval_loader,
                criterion=criterion,
                optimizer=optimizer,
                n_epochs=n_epochs,
                scheduler=scheduler,
                device=device,
                seed=123)

In [ ]:
plot_train_history(history)

In [ ]:
class PerformanceBenchmark:
    """
    Create a performance benchmark for a fine-tuned BERT model and measure various performance metrics.

    This class allows you to evaluate the performance of a fine-tuned BERT model, including:
    - Accuracy of the model using the evaluation set.
    - Size of the model in MiB.
    - Latency of the model in ms.

    Parameters:
        tokenizer (PreTrainedTokenizer): The tokenizer to convert text into tensors.
        model (nn.Module): The fine-tuned model.
        eval_loader (DataLoader): The evaluation DataLoader.
        name (str): The benchmark name.
        device (Union[int, str, None], optional): The device for benchmarking (e.g., 'cuda', 'cpu').
            Default is 'cuda' if available.

    Methods:
        evaluate(): Run evaluation on the eval_loader DataLoader and compute the accuracy for the model.
        compute_size(): Get model size in MiB.
        compute_latency(): Compute mean and standard deviation of latency in ms.
        run_benchmark(): Run all tests and return a dictionary with results.
        compare_benchmarks(*args): Return a Pandas DataFrame with benchmark comparisons.

    The performance benchmark class is designed to measure various aspects of a fine-tuned BERT model's performance.
    """
    def __init__(self, 
                 tokenizer: PreTrainedTokenizer,
                 model: nn.Module,
                 eval_loader: DataLoader, 
                 name: str, 
                 device: Union[int, str, None]=None):
        self.tokenizer = tokenizer
        self.model = model
        self.eval_loader = eval_loader
        self.name = name
        if device is None:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device

        
    def evaluate(self) -> float:
        """
        Run evaluation on the eval_loader DataLoader and compute the accuracy for the model.

        Returns:
            float: The accuracy of the model.
        """
        _, eval_acc = evaluate(model=self.model, 
                               dataloader=self.eval_loader,
                               criterion=nn.CrossEntropyLoss(), # Just to run the function
                               device=self.device)
        return np.round(eval_acc, decimals=3)
    
    
    def compute_size(self) -> float:
        """
        Get model size in MiB.

        Returns:
            float: The model size in MiB.
        """
        param_size, buffer_size = 0, 0
        for param in self.model.parameters():
            param_size += param.nelement() * param.element_size()
        for buffer in self.model.buffers():
            buffer_size += buffer.nelement() * buffer.element_size()
        # Compute total size in MiB
        model_size = (param_size + buffer_size) / (1024**2) # B -> MiB
        return np.round(model_size, decimals=3)
    
    
    def compute_latency(self) -> Tuple[float, float]:
        """
        Compute mean and std latency in ms.

        Returns:
            Tuple[float, float]: The mean and standard deviation of latency in ms.
        """
        sample = "what is the equivalent of, 'life is good' in french"
        tokens = self.tokenizer(sample, return_tensors="pt").to(self.device)
        # Send model to proper device
        self.model.to(self.device)
        # Warmup
        for _ in range(10):
            _ = self.model(**tokens)
        # Timed run
        latencies = []
        for _ in range(100):
            start_time = perf_counter()
            _ = self.model(**tokens)
            end_time = perf_counter()
            latency = end_time - start_time # S
            latencies.append(latency)
        mean_latency = np.mean(latencies) * 1000 # ms
        std_latency = np.std(latencies) * 1000 # ms
        return (np.round(mean_latency, decimals=3),
                np.round(std_latency, decimals=4))
    
    
    def run_benchmark(self) -> Dict:
        """
        Run all tests and return a dictionary with results.
        
        Returns:
            Dict: A dictionary containing benchmark results.
        """
        mean_latency, std_latency = self.compute_latency()
        results = {
            "size": self.compute_size(),
            "mean_latency": mean_latency,
            "std_latency": std_latency,
            "accuracy": self.evaluate()
        }
        return results
            
    
    @staticmethod
    def compare_benchmarks(*args) -> pd.DataFrame:
        """
        Return a Pandas DataFrame with all benchmarks.
        
        Parameters
            *args (PerformanceBenchmark): List or tuple with benchmarks to be compared.
                
        Returns
            pd.DataFrame: Pandas DataFrame with the tests comparisson.
        """
        if not any(isinstance(bench, PerformanceBenchmark) for bench in args):
            raise TypeError("All arguments must be 'PerformanceBenchmark' instances.")
            
        data = pd.DataFrame()
        for bench in args:
            results = {"name": bench.name}
            results.update(bench.run_benchmark())
            results = {k: [v] for k, v in results.items()}
            data = pd.concat([data, pd.DataFrame(results)], axis=0, ignore_index=True)
        return data

In [ ]:
# Creating the benchmark for the current fine-tuned model
benchmark1 = PerformanceBenchmark(tokenizer, model, eval_loader, "bert fine-tune")
PerformanceBenchmark.compare_benchmarks(benchmark1)

In [ ]:
class DistilBERTForClassification(nn.Module):
    """
    DistilBERT model fine-tuned for classification tasks.

    Parameters:
        pretrainedModel (DistilBERT): Pre-trained distilBert model.
        n_labels (int): The number of distinct labels for classification.
    """
    def __init__(self, 
                 pretrainedModel: BertModel,
                 n_labels: int):
        super(DistilBERTForClassification, self).__init__()
        self.config = pretrainedModel.config
        self.hidden_size = self.config.hidden_size
        self.n_labels = n_labels
        
        # Pretrained distilBERT
        self.distilbert = pretrainedModel
        
        # Get config dropout rate
        if self.config.seq_classif_dropout is not None:
            classifier_dropout = self.config.seq_classif_dropout
        else:
            classifier_dropout = 0.1

        # Pre-classification head
        self.pre_classifier = nn.Linear(self.hidden_size, self.hidden_size)
        
        # Classifier head (dropout + linear layer)
        self.dropout = nn.Dropout(classifier_dropout)
        self.linear = nn.Linear(self.hidden_size, self.n_labels)
        
    
    def forward(self, **kwargs):
        """
        Perform a forward pass for the DistilBERT model with the classification head.

        Parameters:
            **kwargs: Keyword arguments to pass to the DistilBERT model. 
                May include the following keys: 'input_ids' and 'attention_mask'.

        Returns:
            torch.Tensor: The output tensor after passing through the classification head.
        """
        outputs = self.distilbert(**kwargs).get("last_hidden_state")[:, 0, :]
        
        # Pre classification
        outputs = self.pre_classifier(outputs)
        outputs = nn.ReLU()(outputs)
        
        # Classifier
        outputs = self.dropout(outputs)
        outputs = self.linear(outputs)
        return outputs

In [ ]:
# Model for sequence classification
distil_model = DistilBERTForClassification(distilbert_model, n_classes).to(device)

In [ ]:
distil_nparams = get_num_parameters(distil_model)
print(f"Distilled Bert has {distil_nparams:,} parameters.")

In [ ]:
# Set seed for reproductivity
set_seed(123)

# New data collator
distil_collate_fn = create_collate_fn(distilbert_tokenizer)

# New train and eval loaders

# Create dataloaders
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=distil_collate_fn)
eval_loader = DataLoader(eval_set, batch_size=32, shuffle=False, collate_fn=distil_collate_fn)

In [ ]:
# Training args
n_epochs = 2
lr = 3e-5 # learning rate
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(distil_model.parameters(), lr=lr)
scheduler = StepLR(optimizer, step_size=1, gamma=.9) # Reduces 10% of the initial lr

# Train the model
history = train(distil_model,
                train_loader,
                eval_loader,
                criterion=criterion,
                optimizer=optimizer,
                n_epochs=n_epochs,
                scheduler=scheduler,
                device=device,
                seed=123)

In [ ]:
# Plot train history
plot_train_history(history)

In [ ]:
# Check distilBert performance
benchmark2 = PerformanceBenchmark(distilbert_tokenizer,
                                  distil_model,
                                  eval_loader,
                                  "distilbert fine-tune")
PerformanceBenchmark.compare_benchmarks(benchmark1, benchmark2)

In [ ]:
# All benchmarks
all_benchmarks = [benchmark1, benchmark2]

In [ ]:
class DistillationLoss(nn.Module):
    def __init__(self, 
                 temperature: float=2.0,
                 alpha: float=0.5):
        """
        Calculates the distillation loss for knowledge transfer from a teacher model to a student model.
        
        The loss consists of a combination of student's cross-entropy loss and Kullback-Leibler divergence loss
        between the student and teacher probability distributions. 
        
        The trade-off between these losses is controlled by the 'alpha' parameter. The final loss is 
        averaged by the number of data points in 'student_logits' for scaling.
        
        temperature (float, optional): The temperature parameter for distillation (default is 2.0).
        alpha (float, optional): The weighting factor for the trade-off between student and teacher losses (default is 0.5).
            Must be in the range [0.0, 1.0].
        """
        super(DistillationLoss, self).__init__()
        self.temperature = temperature
        # Normalizes the alpha between 0.0 and 1.0
        self.alpha = max(0.0, min(1.0, alpha))
        
    
    def forward(self, 
                student_logits: torch.tensor,
                teacher_logits: torch.tensor,
                labels: torch.tensor) -> torch.tensor:
        """
        Compute the distillation loss based on the student's and teacher's logits.
        
        Parameters:
            student_logits (torch.tensor): Logits produced by the student model.
            teacher_logits (torch.tensor): Logits produced by the teacher model.
            labels (torch.tensor): True labels.
            
        Returns:
            torch.tensor: The computed distillation loss.
        """
        # Calculate probabilities
        student_probs = F.softmax(student_logits/self.temperature, dim=-1)
        teacher_probs = F.softmax(teacher_logits/self.temperature, dim=-1)

        # Calculate student's cross entropy loss
        student_loss = nn.CrossEntropyLoss(reduction="mean")(student_logits, labels)

        # Calculate Kullback-Leibler loss between the teacher and student
        kl_loss = nn.KLDivLoss(reduction="batchmean", log_target=True)
        teacher_loss = (self.temperature**2) * kl_loss(student_probs, teacher_probs)

        # Calculate combined loss
        combined_loss = (self.alpha * student_loss) + ((1 - self.alpha) * teacher_loss)

        # Return mean loss
        return combined_loss

In [ ]:
def create_dataloaders(dataset: Dataset,
                       batch_size: int=1,
                       student_collate_fn: Callable=None,
                       teacher_collate_fn: Callable=None,
                       shuffle: bool=True,
                       seed: int=None) -> Tuple[DataLoader, DataLoader]:
    """
    Create data loaders for the student and teacher models.
    
    Parameters:
        dataset (Dataset): The dataset to create data loaders from.
        batch_size (int, optional): The batch size for the data loaders (default is 1).
        student_collate_fn (Callable, optional): A function to collate student model batches.
        teacher_collate_fn (Callable, optional): A function to collate teacher model batches.
        shuffle (bool, optional): Whether to shuffle the data (default is True).
        seed (int, optional): Seed for randomization (default is None).
        
    Returns:
        Tuple[DataLoader, DataLoader]: A tuple containing the data loaders for
            the student and teacher models.
    """
    if seed is not None:
        if isinstance(seed, int):
            set_seed(seed)
        else:
            raise TypeError("seed must be an integer.")
            
    # Create shuffled indices
    indices = list(range(0, len(dataset)))
    if shuffle:
        indices = np.random.choice(indices,
                                   size=len(indices),
                                   replace=False)
    
    # Create DataLoaders
    student_loader = DataLoader([dataset[idx] for idx in indices],
                                batch_size=batch_size,
                                shuffle=False,
                                collate_fn=student_collate_fn)
    teacher_loader = DataLoader([dataset[idx] for idx in indices],
                                batch_size=batch_size,
                                shuffle=False,
                                collate_fn=teacher_collate_fn)
    return student_loader, teacher_loader

In [ ]:
def train_one_epoch_student(student_model: nn.Module,
                            teacher_model: nn.Module, 
                            student_loader: DataLoader, 
                            teacher_loader: DataLoader, 
                            criterion: nn.Module, 
                            optimizer: Optimizer, 
                            device: Union[int, str, None]=None, 
                            seed: Union[int, None]=None) -> float:
    """
    Trains the student model with soft probabilities of the teacher model.
    
    Parameters:
        student_model (nn.Module): The student model to be trained.
        teacher_model (nn.Module): The teacher model for knowledge distillation.
        student_loader (DataLoader): DataLoader for the student model.
        teacher_loader (DataLoader): DataLoader for the teacher model..
        criterion (nn.Module): The custom distillation loss function.
        optimizer (Optimizer): The optimizer used for training.
        device (Union[int, str, None], optional): The device on which to perform training (default is None).

    Returns:
        float: The mean train loss per batch.
    """
    if seed is not None:
        if isinstance(seed, int):
            set_seed(seed)
        else:
            raise TypeError("seed must be an integer.")
            
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    else:
        device = device
            
    student_model.train()
    teacher_model.eval()
    train_loss = 0
    
    # zip is meant to handle iterables with different lenghts, therefore tqdm
    # won't know the 'total' length so we need to specify it.
    for student_batch, teacher_batch in tqdm(zip(student_loader, teacher_loader),
                                             total=len(student_loader),
                                             desc="train", 
                                             leave=False):
        # Zero student gradients
        optimizer.zero_grad()
        
        # Send data to device
        student_inputs = {k: v.to(device) for k, v in student_batch.items() if k != "labels"}
        labels = student_batch["labels"].to(device)
        
        # No need for teacher labels
        teacher_inputs = {k: v.to(device) for k, v in teacher_batch.items() if k != "labels"}
        
        # Compute teacher logits without tracking gradients
        with torch.no_grad():
            teacher_logits = teacher_model(**teacher_inputs)
            
        # Compute student logits
        student_logits = student_model(**student_inputs)
        
        # Compute loss
        loss = criterion(student_logits, teacher_logits, labels)
        train_loss += loss.item()
        
        # Calculate gradients
        loss.backward()
        
        # Adjust weights
        optimizer.step()
        
    # mean loss per batch
    return train_loss/len(student_loader)

In [ ]:
def evaluate_student(student_model: nn.Module,
                     teacher_model: nn.Module,
                     student_loader: DataLoader,
                     teacher_loader: DataLoader, 
                     criterion: nn.Module,
                     device: Union[int, str, None]=None) -> Tuple[float, float]:
    """
    Evaluate the student model's performance by comparing it to the teacher model on a given dataset.
    Return a tuple with evaluation loss and accuracy. The later is based on the most probable
    class predicted by the student model after applying Softmax to its logits.

    Args:
        student_model (nn.Module): The student model to be evaluated.
        teacher_model (nn.Module): The teacher model for reference in knowledge distillation.
        student_loader (DataLoader): DataLoader for the student model.
        teacher_loader (DataLoader): DataLoader for the teacher model.
        criterion (nn.Module): The loss function used for evaluation.
        device (Union[int, str, None], optional): Device for evaluation (e.g., 'cuda', 'cpu'). Default is None.

    Returns:
        Tuple[float, float]: A tuple containing the evaluation loss and accuracy.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    else:
        device = device
    
    # Set models to evaluation mode
    student_model.eval()
    teacher_model.eval()
    
    # Store loss, labels and predictions
    eval_loss = 0
    true_labels, all_preds = [], []
    
    for student_batch, teacher_batch in tqdm(zip(student_loader, teacher_loader),
                                             total=len(student_loader),
                                             desc="eval", 
                                             leave=False):
        # Send data to device
        student_inputs = {k: v.to(device) for k, v in student_batch.items() if k != "labels"}
        labels = student_batch["labels"].to(device)
        
        # No need for teacher labels
        teacher_inputs = {k: v.to(device) for k, v in teacher_batch.items() if k != "labels"}
        
        # Run forward pass without tracking grads
        with torch.no_grad():
            student_logits = student_model(**student_inputs)
            teacher_logits = teacher_model(**teacher_inputs)
        
        # Calculate loss
        eval_loss += criterion(student_logits, teacher_logits, labels).item()
        
        # Apply Softmax to normalize multi-class logits to predictions
        probs = F.softmax(student_logits, dim=-1)
        
        # Select the most probable class
        preds = torch.argmax(probs, dim=-1)
        
        # Store data to calculate accuracy
        all_preds.extend(preds.tolist())
        true_labels.extend(labels.tolist())
    
    # Calculate accuracy
    eval_acc = accuracy_score(true_labels, all_preds)
    # Return mean loss per batch
    return (eval_loss/len(student_loader), eval_acc)

In [ ]:
def distil_train(student_model: nn.Module, 
                 teacher_model: nn.Module,
                 train_set: Dataset,
                 eval_set: Dataset,
                 criterion: nn.Module,
                 optimizer: Optimizer,
                 n_epochs: int=1,
                 batch_size: int=32,
                 student_collate_fn: Callable=None,
                 teacher_collate_fn: Callable=None,
                 scheduler: Union[None, Optimizer]=None,
                 device: Union[None, str, int]=None,
                 seed: Union[None, int]=None) -> Dict[str, List[float]]:
    """
    Train the student model by distillation for a specified number of epochs and
    evaluate its performance on the evaluation set, returning a dictionary with 
    training history.

    Parameters:
        student_model (nn.Module): The student model to be trained.
        teacher_model (nn.Module): The teacher model used for knowledge distillation.
        train_set (Dataset): The training dataset.
        eval_set (Dataset): The evaluation dataset.
        criterion (nn.Module): The loss function used for training.
        optimizer (Optimizer): The optimizer for updating the student model's parameters.
        n_epochs (int, optional): Number of training epochs. Default is 1.
        batch_size (int, optional): Batch size for data loading. Default is 32.
        student_collate_fn (Callable, optional): Custom collate function for student model. Default is None.
        teacher_collate_fn (Callable, optional): Custom collate function for teacher model. Default is None.
        scheduler (Union[None, Optimizer], optional): Learning rate scheduler. Default is None.
        device (Union[None, str, int], optional): Device for training (e.g., 'cuda', 'cpu'). Default is 'cuda' if available.
        seed (Union[None, int], optional): Random seed for reproducibility. Default is None.

    Returns:
        Dict[str, List[float]]: A dictionary containing training history with keys 'train_loss',
        'eval_loss', and 'eval_acc'.
    """
    if seed is not None:
        if isinstance(seed, int):
            set_seed(seed)
        else:
            raise TypeError("seed must be an integer.")
            
    if device is not None:
        device = device
    else: # Infer
        device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # History
    history = {"train_loss": [], "eval_loss": [], "eval_acc": []}
    
    # Save best model
    best_acc = 0
    best_checkpoint = dict()
    
    # Create the evaluation dataloaders
    # Since there's no need to re-shuffle we don't need to recreate
    # the evaluation dataloaders at every epoch.
    student_eval_loader, teacher_eval_loader = create_dataloaders(dataset=eval_set,
                                                                  batch_size=batch_size,
                                                                  student_collate_fn=student_collate_fn,
                                                                  teacher_collate_fn=teacher_collate_fn,
                                                                  shuffle=False, # No need to shuffle
                                                                  seed=seed)
        
    for epoch in tqdm(range(1, n_epochs + 1), desc="train"):
        # Create the training dataloaders on every epoch to ensure re-shuffling
        student_loader, teacher_loader = create_dataloaders(dataset=train_set,
                                                            batch_size=batch_size,
                                                            student_collate_fn=student_collate_fn,
                                                            teacher_collate_fn=teacher_collate_fn, 
                                                            seed=seed)
        
        # Train one epoch and calculate the train time in seconds
        start_time = perf_counter()
        train_loss = train_one_epoch_student(student_model=student_model,
                                             teacher_model=teacher_model,
                                             student_loader=student_loader,
                                             teacher_loader=teacher_loader, 
                                             criterion=criterion, 
                                             optimizer=optimizer,
                                             device=device)
        end_time = perf_counter()
        elapsed_time = end_time - start_time
        
        # Adjust lr
        if scheduler is not None:
            scheduler.step()

        # Run evaluation
        eval_loss, eval_acc = evaluate_student(student_model=student_model, 
                                               teacher_model=teacher_model,
                                               student_loader=student_eval_loader,
                                               teacher_loader=teacher_eval_loader,
                                               criterion=criterion,
                                               device=device)
        
        # Save best model
        is_best = eval_acc > best_acc
        if is_best:
            # Save model's state_dict
            student_model.eval() # Freeze dropout
            best_checkpoint["state_dict"] = deepcopy(student_model.state_dict())
            # Update best eval accuracy
            best_acc = eval_acc
            
        # Updated history
        history["train_loss"].append(train_loss)
        history["eval_loss"].append(eval_loss)
        history["eval_acc"].append(eval_acc)
        
        # Report
        print(f"Epoch {epoch}: Train Loss {train_loss:.3f} - Eval Loss {eval_loss:.3f} - Eval Acc {eval_acc:.3f} - Train Time {elapsed_time:,.1f}")
        
    print("Loading best checkpoint...")
    student_model = student_model.load_state_dict(best_checkpoint["state_dict"])
    
    return history

In [ ]:
# Re-initialize the distilBert with classification head
distil_model = DistilBERTForClassification(distilbert_model, n_classes).to(device)

# Training args
n_epochs = 2
lr = 3e-5 # learning rate
criterion = DistillationLoss()
optimizer = AdamW(distil_model.parameters(), lr=lr)
scheduler = StepLR(optimizer, step_size=1, gamma=.9) # Reduces 10% of the initial lr

# Train the model
history = distil_train(distil_model,
                       model,
                       train_set,
                       eval_set,
                       criterion=criterion,
                       optimizer=optimizer,
                       n_epochs=n_epochs,
                       student_collate_fn=distil_collate_fn,
                       teacher_collate_fn=collate_fn,
                       scheduler=scheduler,
                       device=device,
                       seed=123)

In [ ]:
plot_train_history(history)

In [ ]:
set_seed(123)

# Get student DataLoader
distil_eval_loader, _ = create_dataloaders(dataset=eval_set,
                                           batch_size=32,
                                           student_collate_fn=distil_collate_fn,
                                           shuffle=False)

# Creating the benchmark for the distilled model
benchmark3 = PerformanceBenchmark(distilbert_tokenizer,
                                  distil_model,
                                  distil_eval_loader,
                                  "distilbert-kd") # KD = 'Knowledge Distillation'

# Add new benchmark to all_benchmarks list
all_benchmarks.append(benchmark3)

# Comparing benchmarks
PerformanceBenchmark.compare_benchmarks(*all_benchmarks)

In [ ]:
from torch.quantization import quantize_dynamic

# Copy the distilled model
distil_model_quantized = deepcopy(distil_model).to("cpu")

# Quantize model
distil_model_quantized = quantize_dynamic(distil_model_quantized,
                                          {nn.Linear}, # Specify the modules to be quantized
                                          dtype=torch.qint8) # Specify the precision

In [ ]:
benchmark4 = PerformanceBenchmark(distilbert_tokenizer,
                                  distil_model_quantized,
                                  distil_eval_loader,
                                  "distilbert-kd-quantized", 
                                  device="cpu")

# Add new benchmark to all_benchmarks list
all_benchmarks.append(benchmark4)

In [ ]:
# Setting all benchmarks to 'cpu' for a just comparisson
for bench in all_benchmarks:
    bench.device = "cpu"

PerformanceBenchmark.compare_benchmarks(*all_benchmarks)

# Zynthe: End-to-End Knowledge Distillation (Text)
# This notebook runs a transparent KD pipeline with robust preprocessing, training, evaluation, and a simple inference helper.
